<a href="https://colab.research.google.com/github/Rasmy-r7/Research/blob/main/cassandra_Preprocess.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openpyxl -q

import pandas as pd
import re
from google.colab import files

print("✅ Ready!")

✅ Ready!


In [2]:
print("📁 Upload cassandra_bugs.csv")
uploaded = files.upload()

df = pd.read_csv('cassandra_bugs.csv', low_memory=False)
print(f"✅ Loaded: {df.shape}")
print("\nOriginal Priority distribution:")
print(df['Priority'].value_counts(dropna=False))

📁 Upload cassandra_bugs.csv


Saving cassandra_bugs.csv to cassandra_bugs.csv
✅ Loaded: (4612, 8)

Original Priority distribution:
Priority
Normal    4293
Low        193
Urgent      68
High        58
Name: count, dtype: int64


In [3]:
# Cassandra is Jira-based (not Bugzilla)
# Jira standard: Urgent→High, High→High, Normal→Medium, Low→Low

jira_map = {
    'Urgent': 'High',
    'High':   'High',
    'Normal': 'Medium',
    'Low':    'Low'
}

df['priority'] = df['Priority'].map(jira_map)

# Drop unmapped rows if any
df = df[df['priority'].notna()].copy()
df = df.reset_index(drop=True)

print(f"✅ After fixing Priority: {df.shape}")
print(df['priority'].value_counts())

✅ After fixing Priority: (4612, 9)
priority
Medium    4293
Low        193
High       126
Name: count, dtype: int64


In [4]:
df['text'] = (
    df['Summary'].fillna('').str.strip() + ' ' +
    df['Description'].fillna('').str.strip()
)

print(f"✅ Text column created!")
print(f"Avg words (before clean): {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words (before clean): {df['text'].str.split().str.len().max()}")

✅ Text column created!
Avg words (before clean): 124
Max words (before clean): 10626


In [5]:
def clean_text(text):
    text = str(text)

    # Remove log lines [task 2020-01-02T14:00:26Z]
    text = re.sub(
        r'\[task \d{4}-\d{2}-\d{2}T[\d:.]+Z\].*?(?=\[task|\Z)',
        '', text, flags=re.DOTALL
    )
    # Remove Jira/Bugzilla boilerplate
    text = re.sub(
        r'(User Agent|Build Identifier|Filed by|Parsed log|Full log)\s*:.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove test log lines
    text = re.sub(
        r'(TEST-PASS|TEST-OK|TEST-FAIL|TEST-UNEXPECTED|GECKO\(\d+\)|INFO\s*-)\s*.*?(?=\n|$)',
        '', text, flags=re.IGNORECASE
    )
    # Remove stack traces (at com.example... / at java.lang...)
    text = re.sub(
        r'\s+at\s+[\w\.\$]+\([\w\.\:]+\)',
        '', text, flags=re.IGNORECASE
    )
    # Remove memory stat lines
    text = re.sub(r'MEMORY STAT.*?(?=\n|$)', '', text, flags=re.IGNORECASE)
    # Remove URLs
    text = re.sub(r'http[s]?://\S+', '', text)
    text = re.sub(r'www\.\S+', '', text)
    # Remove file paths
    text = re.sub(
        r'[\w/\-\.]+\.(js|cpp|py|java|html|css|ts|json|xml|md|log|yaml|conf)\b',
        '', text, flags=re.IGNORECASE
    )
    # Remove code blocks
    text = re.sub(r'```.*?```', ' ', text, flags=re.DOTALL)
    text = re.sub(r'`[^`\n]+`', ' ', text)
    # Remove markdown bold/italic
    text = re.sub(r'\*\*([^*]+)\*\*', r'\1', text)
    text = re.sub(r'\*([^*\n]+)\*', r'\1', text)
    # Remove markdown links [text](url)
    text = re.sub(r'\[([^\]]+)\]\([^\)]*\)', r'\1', text)
    # Remove hex values
    text = re.sub(r'\b0x[0-9a-fA-F]+\b', '', text)
    # Remove timestamps
    text = re.sub(r'\b\d{2}:\d{2}:\d{2}(\.\d+)?\b', '', text)
    # Remove dates
    text = re.sub(r'\b\d{4}-\d{2}-\d{2}\b', '', text)
    # Remove version numbers
    text = re.sub(r'\bv?\d+\.\d+(\.\d+)*\b', '', text)
    # Remove separator lines
    text = re.sub(r'[-=*_#~^|<>]{3,}', ' ', text)

    # Remove noisy lines
    lines = text.split('\n')
    clean_lines = []
    for line in lines:
        line = line.strip()
        if len(line) < 3:
            continue
        alpha_count = sum(c.isalpha() for c in line)
        if alpha_count < 3:
            continue
        if alpha_count / (len(line) + 1) < 0.3:
            continue
        clean_lines.append(line)

    text = ' '.join(clean_lines)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("⏳ Cleaning texts — please wait...")
df['text'] = df['text'].apply(clean_text)
print("✅ Text cleaning done!")
print(f"Avg words (after clean): {df['text'].str.split().str.len().mean():.0f}")

⏳ Cleaning texts — please wait...
✅ Text cleaning done!
Avg words (after clean): 114


In [6]:
def truncate(text, max_words=100):
    return ' '.join(text.split()[:max_words])

df['text'] = df['text'].apply(truncate)

print("✅ Truncated to max 100 words!")
print(f"Avg words: {df['text'].str.split().str.len().mean():.0f}")
print(f"Max words: {df['text'].str.split().str.len().max()}")

✅ Truncated to max 100 words!
Avg words: 65
Max words: 100


In [7]:
df = df[df['text'].str.split().str.len() >= 5].copy()
df = df.drop_duplicates(subset='text').copy()
df = df.reset_index(drop=True)

print(f"✅ After quality filter: {df.shape}")
print(df['priority'].value_counts())

✅ After quality filter: (4560, 10)
priority
Medium    4249
Low        189
High       122
Name: count, dtype: int64


In [8]:
label_map = {'High': 0, 'Medium': 1, 'Low': 2}
df['priority_id'] = df['priority'].map(label_map)
df['source']      = 'gitbugs_cassandra'

final = df[['text', 'priority', 'priority_id', 'source']].copy()

print("✅ Final columns set!")
print(final.head(3).to_string())

✅ Final columns set!
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           text priority  priority_id             source
0                                                         

In [9]:
wc = final['text'].str.split().str.len()

print("=" * 50)
print("   ✅ CASSANDRA_CLEAN — FINAL RESULTS")
print("=" * 50)
print(f"  Total rows     : {len(final):,}")
print(f"  Avg word count : {wc.mean():.0f}")
print(f"  Min word count : {wc.min()}")
print(f"  Max word count : {wc.max()}")
print()
print("  Priority distribution:")
print(final['priority'].value_counts().to_string())
print()
print("  Priority %:")
print((final['priority'].value_counts(normalize=True)*100).round(1).to_string())
print()
print("  Sample rows:")
for _, row in final.sample(3, random_state=42).iterrows():
    print(f"\n  [{row['priority']}] {row['text'][:120]}")
print()
print("=" * 50)

final.to_csv('cassandra_clean.csv', index=False)
files.download('cassandra_clean.csv')
print("✅ Downloaded: cassandra_clean.csv")

   ✅ CASSANDRA_CLEAN — FINAL RESULTS
  Total rows     : 4,560
  Avg word count : 66
  Min word count : 5
  Max word count : 100

  Priority distribution:
priority
Medium    4249
Low        189
High       122

  Priority %:
priority
Medium    93.2
Low        4.1
High       2.7

  Sample rows:

  [Medium] Expose decommission state to nodetool info Currently, when a node is being decommissioned and if any failure happens, th

  [Medium] Secure HTTPS link to the research paper in the website landing page In [ , under the Performant section, there's a link 

  [Medium] Doc update: stream-entire-sstable supports all compaction strategies and internode encryption As [~mck] point out, doc n



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: cassandra_clean.csv
